# 7.6 Gaussian Mixture Models (GMM)

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 7: Clustering Techniques

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-07-clustering-techniques/07.06_gaussian-mixture-models.ipynb)

Companion code for Section 7.6 of the book. Run the cells in order from top to bottom.

**Example 7.6.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

# ── Generate overlapping clusters (GMM's strength) ────────────────
X, y_true = make_blobs(n_samples=500, centers=3, cluster_std=1.5, random_state=42)
X = StandardScaler().fit_transform(X)

# ── Fit GMM with different covariance types ───────────────────────
cov_types = ['full', 'tied', 'diag', 'spherical']
print('GMM Covariance Type Comparison (k=3):')
print(f'{"Covariance Type":<18} {"AIC":>8} {"BIC":>8} {"ARI":>8} {"Log-Lik":>10}')
print('='*58)

results = {}
for cov_type in cov_types:
    gmm = GaussianMixture(n_components=3, covariance_type=cov_type,
                          n_init=10, random_state=42)
    gmm.fit(X)
    labels = gmm.predict(X)
    ari    = adjusted_rand_score(y_true, labels)
    results[cov_type] = {'gmm': gmm, 'labels': labels, 'ari': ari}
    print(f'{cov_type:<18} {gmm.aic(X):>8.2f} {gmm.bic(X):>8.2f} {ari:>8.4f} {gmm.lower_bound_:>10.4f}')

# ── Model selection: BIC for choosing k ──────────────────────────
print('\nSelecting number of components via BIC:')
print(f'{"k":>4} {"AIC":>10} {"BIC":>10} {"BIC diff":>10}')
print('='*40)
bics, aics = [], []
k_range = range(1, 9)
for k in k_range:
    gmm_k = GaussianMixture(n_components=k, covariance_type='full',
                             n_init=10, random_state=42)
    gmm_k.fit(X)
    bics.append(gmm_k.bic(X))
    aics.append(gmm_k.aic(X))

for i, (k, bic) in enumerate(zip(k_range, bics)):
    diff = bic - bics[i-1] if i > 0 else 0
    marker = ' ← optimal' if bic == min(bics) else ''
    print(f'{k:>4} {aics[i]:>10.2f} {bic:>10.2f} {diff:>10.2f}{marker}')

# ── Best GMM: soft assignments ─────────────────────────────────────
gmm_best = GaussianMixture(n_components=3, covariance_type='full',
                            n_init=10, random_state=42)
gmm_best.fit(X)
probs  = gmm_best.predict_proba(X)   # Soft probabilities
labels = gmm_best.predict(X)          # Hard assignments

print(f'\nBest GMM (full covariance, k=3):')
print(f'  ARI:            {adjusted_rand_score(y_true, labels):.4f}')
print(f'  Log-likelihood: {gmm_best.lower_bound_:.4f}')
print(f'  Mixing weights: {gmm_best.weights_.round(3)}')
print(f'\nSoft assignment probabilities for first 5 points:')
print(f'{"Point":>6} {"P(C0)":>8} {"P(C1)":>8} {"P(C2)":>8} {"Hard Label":>12}')
for i in range(5):
    p = probs[i]
    print(f'{i:>6} {p[0]:>8.4f} {p[1]:>8.4f} {p[2]:>8.4f} {labels[i]:>12}')

# ── Visualisation ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Gaussian Mixture Model Analysis', fontsize=14, fontweight='bold')
palette = ['#E53935','#1976D2','#43A047']

# Hard assignments
ax = axes[0]
for k in range(3):
    mask = labels == k
    ax.scatter(X[mask,0], X[mask,1], s=15, alpha=0.7, color=palette[k])
ax.scatter(gmm_best.means_[:,0], gmm_best.means_[:,1], s=300,
          c='black', marker='*', zorder=10, edgecolors='white', lw=1.5,
          label='Component means')
ax.set_title('GMM Hard Assignments'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Uncertainty: max probability
ax = axes[1]
max_prob = probs.max(axis=1)
sc = ax.scatter(X[:,0], X[:,1], c=max_prob, cmap='RdYlGn',
               s=20, alpha=0.8, vmin=0.5, vmax=1.0)
plt.colorbar(sc, ax=ax, label='Max cluster probability')
ax.set_title('Soft Assignment Confidence\n(green=certain, red=uncertain)')
ax.grid(alpha=0.3)

# BIC curve
ax = axes[2]
ax.plot(k_range, bics,  'r-o', ms=8, lw=2, label='BIC')
ax.plot(k_range, aics,  'b-o', ms=8, lw=2, label='AIC')
ax.axvline(x=3, color='green', ls='--', lw=2, label='Optimal k=3')
ax.set_xlabel('Number of Components (k)')
ax.set_ylabel('Information Criterion')
ax.set_title('BIC/AIC for Selecting k\n(Minimum = optimal)')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('gmm_analysis.png', dpi=150, bbox_inches='tight')
plt.show()